# Chapter 7 - Unsupervised Learning


**Nama:** Aryo Zaki Abdillah  
**NIM:** 101032300060  
**Kelas:** TK47-02  
**Universitas:** Telkom University

> Reproduksi kode dan pendalaman teoretis dari *Practical Statistics for Data Scientists* (O'Reilly), edisi ke-2.
>
> Struktur topik mengikuti notebook Python chapter 1–7 pada repository resmi:
> https://github.com/gedeck/practical-statistics-for-data-scientists

## Tujuan

Chapter ini membahas Unsupervised Learning, yaitu kumpulan teknik statistik untuk menemukan struktur, pola, atau kelompok dalam data tanpa menggunakan variabel target label.
Topik yang dibahas meliputi Principal Components Analysis (PCA), K-Means Clustering, Hierarchical Clustering,
Model-Based Clustering (Gaussian Mixture Models), serta tantangan pemrosesan variabel kategorikal dan skala data.

## Imports dan Setup Data

Mengimpor modul Python yang dibutuhkan dan memuat dataset Iris untuk demonstrasi teknik Unsupervised Learning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.cluster.hierarchy as sch

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

np.random.seed(42)

## 1. Principal Components Analysis (PCA)

PCA adalah teknik reduksi dimensi linear yang mentransformasikan variabel yang berkorelasi menjadi kumpulan komponen utama ortogonal yang menangkap varians maksimum dalam data.

In [ ]:
iris = load_iris()
X_iris = iris.data
feature_iris = iris.feature_names

# Standardisasi
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Melakukan PCA
pca = PCA()
X_pca = pca.fit_transform(X_iris_scaled)

exp_var = pca.explained_variance_ratio_
cum_var = np.cumsum(exp_var)

# Scree Plot
plt.figure(figsize=(7, 4))
plt.bar(range(1, len(exp_var) + 1), exp_var, alpha=0.6, label="Individual Variance")
plt.step(range(1, len(cum_var) + 1), cum_var, where="mid", label="Cumulative Variance", color="red")
plt.xlabel("Komponen Utama (PC)")
plt.ylabel("Rasio Varians yang Dijelaskan")
plt.title("Scree Plot - Principal Component Analysis")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. K-Means Clustering

K-Means membagi data menjadi K kluster dengan meminimalkan Within-Cluster Sum of Squares (WSS/Inertia). Penentuan jumlah kluster K dilakukan menggunakan Elbow Method dan Silhouette Score.

In [ ]:
wss = []
silhouette_scores = []
k_values = range(2, 9)

for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_iris_scaled)
    wss.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_iris_scaled, labels))

fig, ax1 = plt.subplots(figsize=(8, 4))

ax1.plot(k_values, wss, marker='o', color='tab:blue')
ax1.set_xlabel("Jumlah Kluster (K)")
ax1.set_ylabel("WSS / Inertia", color='tab:blue')

ax2 = ax1.twinx()
ax2.plot(k_values, silhouette_scores, marker='s', color='tab:red', linestyle='--')
ax2.set_ylabel("Silhouette Score", color='tab:red')

plt.title("K-Means: Evaluasi Elbow Method & Silhouette Score")
plt.grid(True, alpha=0.3)
plt.show()

## 3. Hierarchical Clustering

Hierarchical Clustering membangun struktur kluster bertingkat berbentuk pohon dendrogram tanpa harus menentukan jumlah K terlebih dahulu.

In [ ]:
plt.figure(figsize=(10, 5))
dendrogram = sch.dendrogram(sch.linkage(X_iris_scaled, method='ward'))
plt.title("Dendrogram Hierarchical Clustering (Ward Linkage)")
plt.xlabel("Indeks Observasi")
plt.ylabel("Jarak Euklides")
plt.show()

# Pengelompokan dengan 3 kluster
hc = AgglomerativeClustering(n_clusters=3, metric='euclidean', linkage='ward')
hc_labels = hc.fit_predict(X_iris_scaled)

## 4. Model-Based Clustering (Gaussian Mixture Models)

Model-Based Clustering mengasumsikan bahwa data dibangkitkan oleh campuran beberapa distribusi probabilitas, umumnya Gaussian Multivariat, dan dioptimalkan dengan algoritma Expectation-Maximization (EM).

In [ ]:
bic_scores = []
n_components_range = range(1, 7)

for n in n_components_range:
    gmm = GaussianMixture(n_components=n, random_state=42)
    gmm.fit(X_iris_scaled)
    bic_scores.append(gmm.bic(X_iris_scaled))

plt.figure(figsize=(7, 4))
plt.plot(n_components_range, bic_scores, marker='o', color='purple')
plt.xlabel("Jumlah Komponen")
plt.ylabel("Skor BIC (Nilai Lebih Rendah Lebih Baik)")
plt.title("Pemilihan Model GMM berdasarkan BIC")
plt.grid(True, alpha=0.3)
plt.show()

## 5. Scaling dan Pertimbangan Variabel Kategorikal

Algoritma berbasis jarak seperti K-Means dan Hierarchical Clustering sangat sensitif terhadap skala fitur. Variabel tanpa standardisasi akan mendominasi perhitungan jarak.

In [ ]:
# Perbandingan performa clustering sebelum dan sesudah scaling
km_raw = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_iris)
km_scaled = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_iris_scaled)

print("Inertia data mentah (tanpa scaling):", km_raw.inertia_)
print("Inertia data terstandarisasi (dengan scaling):", km_scaled.inertia_)

## Interpretasi

Unsupervised Learning pada Chapter 7 memberikan alat untuk eksplorasi struktur tersembunyi. PCA berhasil mereduksi dimensi data sambil mempertahankan sebagian besar informasi varians. Metode clustering seperti K-Means, Hierarchical Clustering, dan GMM terbukti sensitif terhadap skala data, sehingga standardisasi merupakan tahap eksplorasi awal yang krusial.

## Kesimpulan Chapter 7

Unsupervised Learning membantu memahami pengelompokan alami dan reduksi variabel tanpa membutuhkan label. Pemilihan jumlah kluster dan metode transformasi data sangat mempengaruhi kualitas segmentasi yang dihasilkan.